# A large scale analysis of conversation data across myriad spoken corpora

In [1]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_LOC = 'data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

# path to save data to on completion
VIS_PATH = os.path.join(DATA_LOC, 'web_vis')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join(VIS_PATH, 'final-document.tsv')

In [3]:
def grab_all_files(PATH, file_type='.csv'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

In [4]:
fs = grab_all_files(DATA_PATH, '.tsv')
fs

['data/lme_ready/cha-croatian-ceda-analysis.tsv',
 'data/lme_ready/cha-yiddish-ceda-analysis.tsv',
 'data/lme_ready/cha-callfriend-ceda-analysis.tsv',
 'data/lme_ready/cha-callhome-ceda-analysis.tsv',
 'data/lme_ready/cha-finchat_corpus-ceda-analysis.tsv',
 'data/lme_ready/cha-ko_corpus-ceda-analysis.tsv']

Iteratively import dataframes and editing them

In [5]:
start_from_turn = 5

In [6]:
df = []

In [7]:
for f in tqdm(fs):
    df += [pd.read_table(f, sep='\t')]
    
    if ('/cha-' in f) or ('grace-' in f):
        df[-1]['conversation_id'] = ['-'.join(sp.split('-')[:-1]) for sp in tqdm(df[-1]['x_speaker'])]
   
    elif '/candor-' in f:
        df[-1]['conversation_id'] = [sp.replace('.csv', '') for sp in tqdm(df[-1]['conversation_id'])]
        df[-1]['corpus'] = 'CANDOR'
    
    else:
        if 'conversation_id' in df[-1].columns:
            0
        else:
            df[-1]['conversation_id'] = f
        
    df[-1] = df[-1].loc[
        (df[-1]['nx'] >= 5) 
        & (df[-1]['ny'] >= 5) 
        & (df[-1]['x_turn_id'] >= start_from_turn)
    ]
    
    df[-1]['dyad'] = df[-1]['x_speaker'] + '-' + df[-1]['y_speaker']
    df[-1]['turn_delta'] = (df[-1]['y_turn_id'] - df[-1]['x_turn_id'])
    df[-1] = df[-1][['Hxy', 'nx', 'ny', 'turn_delta', 'self', 'dyad', 'corpus', 'conversation_id']]
    

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/3311067 [00:00<?, ?it/s]

  0%|          | 0/13121 [00:00<?, ?it/s]

  0%|          | 0/11049978 [00:00<?, ?it/s]

  0%|          | 0/11122234 [00:00<?, ?it/s]

  0%|          | 0/92605 [00:00<?, ?it/s]

  0%|          | 0/2190990 [00:00<?, ?it/s]

In [8]:
df = pd.concat(df, ignore_index=True)
df.shape

(16053286, 8)

In [9]:
# to rename the corpus correctly . . . 
def lang(x):
    return x.split('-')[1]

In [10]:
df['lang'] = [lang(x) for x in tqdm(df['corpus'])]

  0%|          | 0/16053286 [00:00<?, ?it/s]

I had some worries about using the japanese dataset with `xlm-roberta-base`. [Some other users have noted that xlm-roberta pathologically renders CoS (thus CoE values) superficially high (low)](https://huggingface.co/FacebookAI/xlm-roberta-base/discussions/18) when comparing across sentences (in other words, the model exhibits high anisotropy for Japanese in particular). 

In [11]:
df = df.loc[~df['lang'].isin(['jpn'])]
df.shape

(14062796, 9)

In [12]:
df['conversation_id'].nunique()

1285

In [13]:
df.head()

,Hxy,nx,ny,turn_delta,self,dyad,corpus,conversation_id,lang
0,2.765743,12.0,10.0,2.0,False,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,cro
1,2.836436,12.0,7.0,6.0,False,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,cro
2,2.906804,12.0,8.0,9.0,False,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,cro
3,3.063219,12.0,5.0,10.0,False,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,cro
4,2.971834,12.0,9.0,12.0,False,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,cro


## Convert values to numeric tags

In [14]:
convert_to_numeric_id = [
    #'x_speaker', 'y_speaker', 
    'dyad', 
    'lang'
    #'x_speaker_turn'
]

In [15]:
for col in convert_to_numeric_id:
    vals = np.unique(df[col].values)
    conversion = {k: i+1 for i,k in enumerate(np.random.choice(vals, size=(len(vals),), replace=False))}
    
    if isinstance(col,list):
        for c  in col:
            df[c] = [conversion[v] for v in tqdm(df[c])]
    
    else:
        df[col] = [conversion[v] for v in tqdm(df[col])]
    

  0%|          | 0/14062796 [00:00<?, ?it/s]

  0%|          | 0/14062796 [00:00<?, ?it/s]

## LME Regression: Predicting CE

In [16]:
from tqdm import tqdm
import statsmodels.formula.api as smf

In [17]:
##########################################
## Main model
##########################################
model = "Hxy ~ nx + ny + turn_delta*self + (1|lang)"
##########################################

In [18]:
start = dt.now()
# md = smf.mixedlm(model, data=df, groups=df['x_speaker_turn'])
md = smf.mixedlm(model, data=df, groups=df['dyad'])
mdf = md.fit()
print('completed in:', dt.now()-start)

completed in: 0:01:08.106640


In [19]:
# mdf.summary()

In [20]:
reporting = pd.DataFrame()
reporting['coefs'] = mdf.params
reporting['stat'] = mdf.tvalues
reporting['p'] = mdf.pvalues
reporting['CI[.025, .975]'] = ['[{}]'.format(', '.join([np.format_float_scientific(x, precision=2) for x in ci.tolist()])) for ci in mdf.conf_int().values]

reporting['coefs'] = reporting['coefs'].apply(lambda x: np.format_float_scientific(x, precision=2))
reporting['stat'] = reporting['stat'].apply(lambda x: np.format_float_scientific(x, precision=2))
reporting['p'] = reporting['p'].apply(lambda x: np.format_float_scientific(x, precision=2))

reporting.head(100)

,coefs,stat,p,"CI[.025, .975]"
Intercept,-1.87e-01,-1.62e+01,4.74e-59,"[-2.09e-01, -1.64e-01]"
self[T.True],-1.08e-01,-1.24e+01,3.43e-35,"[-1.25e-01, -9.1e-02]"
nx,2.87e-01,1.96e+04,0.e+00,"[2.87e-01, 2.87e-01]"
ny,-1.7e-02,-1.15e+03,0.e+00,"[-1.7e-02, -1.69e-02]"
turn_delta,4.40e-04,5.14e+01,0.e+00,"[4.23e-04, 4.57e-04]"
turn_delta:self[T.True],4.14e-04,3.81e+01,0.e+00,"[3.93e-04, 4.35e-04]"
1 | lang,-3.29e-02,-1.8e+01,2.56e-72,"[-3.65e-02, -2.93e-02]"
Group Var,7.17e-01,5.97e+01,0.e+00,"[6.94e-01, 7.41e-01]"


Saving results for reporting

In [21]:
lme_results_name = os.path.join(DATA_LOC, 'lme-results.csv')

In [22]:
reporting.to_csv(lme_results_name, encoding='utf-8')

In [23]:
reporting['Var'] = reporting.index.values

with open(lme_results_name.replace('.csv', '.txt'), 'w') as f:
    txt =  reporting[['Var', 'coefs', 'stat', 'p']].to_latex(index=False).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')

    txt = txt.replace('\\\\', '\\\\\hline').replace('|', '$|$').replace('_delta', ' $\Delta$')
    f.write(txt)
    f.close()

## Create a resid

In [24]:
##########################################
## A set of resids to show ∇H / t
##########################################

model = "Hxy ~ nx + ny + (1|lang)"
##########################################

In [25]:
start = dt.now()
# md = smf.mixedlm(model, data=df, groups=df['x_speaker_turn'])
md = smf.mixedlm(model, data=df, groups=df['dyad'])
mdf = md.fit()
print('completed in:', dt.now()-start)

completed in: 0:00:45.430576


In [26]:
# mdf.summary()
mdf.bse

Intercept    0.010818
nx           0.000015
ny           0.000015
1 | lang     0.001834
Group Var    0.012167
dtype: float64

In [27]:
df['resid'] = mdf.resid

In [28]:
df.to_csv(FINAL_DOC, index=False, encoding='utf-8', sep='\t')